# SMD Data Exploration

Explore the Server Machine Dataset (SMD) used for TranAD multivariate anomaly detection.
This notebook examines machine-1-1 as a representative example.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "code":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "pyproject.toml").exists()
sys.path.insert(0, str(PROJECT_ROOT))

import logging
logging.getLogger("src").setLevel(logging.WARNING)

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

from src.preprocess import load_processed_data

## Load Raw SMD Data

Load the raw CSV data for machine-1-1 using `np.genfromtxt`. The SMD dataset has 38 sensor features per machine.

In [ ]:
MACHINE = "machine-1-1"

train_raw = np.genfromtxt(
    PROJECT_ROOT / "data" / "smd" / "raw" / "train" / f"{MACHINE}.txt",
    delimiter=",",
)
test_raw = np.genfromtxt(
    PROJECT_ROOT / "data" / "smd" / "raw" / "test" / f"{MACHINE}.txt",
    delimiter=",",
)

print(f"Train shape: {train_raw.shape}")
print(f"Test shape:  {test_raw.shape}")
print()
print("Train basic statistics:")
print(f"  Min:  {train_raw.min():.4f}")
print(f"  Max:  {train_raw.max():.4f}")
print(f"  Mean: {train_raw.mean():.4f}")
print(f"  Std:  {train_raw.std():.4f}")
print(f"  NaNs: {np.isnan(train_raw).sum()}")
print()
print("Test basic statistics:")
print(f"  Min:  {test_raw.min():.4f}")
print(f"  Max:  {test_raw.max():.4f}")
print(f"  Mean: {test_raw.mean():.4f}")
print(f"  Std:  {test_raw.std():.4f}")
print(f"  NaNs: {np.isnan(test_raw).sum()}")

## Training Data Heatmap

Visualize all 38 features over the full training period. Each row is a feature, each column is a timestep.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
im = ax.imshow(train_raw.T, aspect="auto", cmap="viridis", interpolation="nearest")
ax.set_xlabel("Timestep")
ax.set_ylabel("Feature Index")
ax.set_title("Machine 1-1: Training Data (38 Features x Time)")
plt.colorbar(im, ax=ax, label="Raw value")
plt.tight_layout()
plt.show()

## Test Data with Ground Truth Anomalies

Plot 4 selected features from the test set with anomaly labels overlaid as red shading.

In [ ]:
test_labels_raw = np.genfromtxt(
    PROJECT_ROOT / "data" / "smd" / "raw" / "test_label" / f"{MACHINE}.txt",
    delimiter=",",
)

print(f"Test labels shape: {test_labels_raw.shape}")
print(f"Anomaly rate: {test_labels_raw.mean():.2%} of test timesteps")

# Select 4 features that show interesting variation
selected_features = [0, 5, 18, 24]

fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
for ax, fidx in zip(axes, selected_features):
    ax.plot(test_raw[:, fidx], linewidth=0.5, color="steelblue")
    ax.fill_between(
        range(len(test_raw)),
        0,
        1,
        where=test_labels_raw > 0,
        alpha=0.2,
        color="red",
        transform=ax.get_xaxis_transform(),
        label="Anomaly" if fidx == selected_features[0] else None,
    )
    ax.set_ylabel(f"Feature {fidx}", fontsize=9)

axes[0].set_title(f"{MACHINE} -- Test Data with Ground Truth Anomalies")
axes[0].legend(loc="upper right", fontsize=8)
axes[-1].set_xlabel("Timestep")
plt.tight_layout()
plt.show()

## Train vs Test Distribution Comparison

Side-by-side boxplots for a subset of 8 features to compare the train and test distributions.

In [ ]:
subset_features = [0, 4, 8, 12, 18, 24, 30, 35]

fig, axes = plt.subplots(1, len(subset_features), figsize=(18, 5), sharey=False)
for ax, fidx in zip(axes, subset_features):
    bp = ax.boxplot(
        [train_raw[:, fidx], test_raw[:, fidx]],
        tick_labels=["Train", "Test"],
        widths=0.6,
        patch_artist=True,
    )
    bp["boxes"][0].set_facecolor("lightblue")
    bp["boxes"][1].set_facecolor("lightsalmon")
    ax.set_title(f"Feature {fidx}", fontsize=9)
    ax.tick_params(axis="both", labelsize=7)

fig.suptitle(f"{MACHINE} -- Train vs Test Distribution", fontsize=13)
plt.tight_layout()
plt.show()

## Feature Correlation Matrix

Compute the Pearson correlation matrix on training data. Correlated feature groups reveal the multivariate structure that TranAD learns to model.

In [ ]:
# Filter out constant features (zero variance) to avoid NaN in correlation
feature_std = train_raw.std(axis=0)
varying_features = np.where(feature_std > 0)[0]
n_constant = train_raw.shape[1] - len(varying_features)
if n_constant > 0:
    print(f"Note: {n_constant} constant feature(s) excluded from correlation (zero variance)")

corr = np.corrcoef(train_raw[:, varying_features].T)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xlabel("Feature Index")
ax.set_ylabel("Feature Index")
ax.set_title(f"{MACHINE} -- Feature Correlation Matrix (Training Data)")
plt.colorbar(im, ax=ax, label="Pearson correlation")
plt.tight_layout()
plt.show()

n_features = corr.shape[0]
n_high_corr = ((np.abs(corr) > 0.7) & (np.eye(n_features) == 0)).sum() // 2
print(f"Highly correlated feature pairs (|r| > 0.7): {n_high_corr}")

## Zoom into Anomaly Segments

Load the processed per-feature interpretation labels and zoom into a segment where multiple features are flagged as anomalous.

In [ ]:
interp_labels = np.load(
    PROJECT_ROOT / "data" / "smd" / "processed" / f"{MACHINE}_interp_labels.npy"
)
print(f"Interp labels shape: {interp_labels.shape}")
print(f"Features with anomalies: {(interp_labels.sum(axis=0) > 0).sum()} of {interp_labels.shape[1]}")

# Find anomaly segment boundaries
anomaly_binary = (interp_labels.sum(axis=1) > 0).astype(int)
padded = np.concatenate([[0], anomaly_binary, [0]])
diffs = np.diff(padded)
starts = np.where(diffs == 1)[0]
ends = np.where(diffs == -1)[0] - 1

print(f"\nTotal anomaly segments: {len(starts)}")

# Find a segment where multiple features are anomalous
best_seg = 0
best_count = 0
for i in range(len(starts)):
    s, e = starts[i], ends[i]
    n_anom_features = (interp_labels[s : e + 1].sum(axis=0) > 0).sum()
    if n_anom_features > best_count:
        best_count = n_anom_features
        best_seg = i

s, e = starts[best_seg], ends[best_seg]
root_features = np.where(interp_labels[s : e + 1].sum(axis=0) > 0)[0]
print(f"\nSelected segment {best_seg + 1}: timesteps [{s}-{e}] (length={e - s + 1})")
print(f"Root cause features: {root_features.tolist()}")

# Plot the anomalous features in this segment with context
context = 100
view_s = max(0, s - context)
view_e = min(len(test_raw), e + context + 1)
features_to_show = root_features[:6]  # cap at 6 for readability

fig, axes = plt.subplots(
    len(features_to_show), 1,
    figsize=(14, 2.5 * len(features_to_show)),
    sharex=True,
)
if len(features_to_show) == 1:
    axes = [axes]

for ax, fidx in zip(axes, features_to_show):
    ax.plot(
        range(view_s, view_e),
        test_raw[view_s:view_e, fidx],
        linewidth=1,
        color="steelblue",
    )
    ax.axvspan(s, e, alpha=0.2, color="red")
    ax.set_ylabel(f"Feature {fidx}\n(ROOT CAUSE)", fontsize=8, color="red")

axes[0].set_title(
    f"Anomaly Segment {best_seg + 1}: timesteps [{s}-{e}] -- {len(root_features)} anomalous features"
)
axes[-1].set_xlabel("Timestep")
plt.tight_layout()
plt.show()

## Why Multivariate Anomaly Detection?

A single-feature spike might be perfectly normal under heavy load -- for example, CPU utilization
jumping to 95% during a batch job is expected. But the **same spike when other features are idle**
(low network I/O, low disk throughput, no active connections) is suspicious: it suggests a
runaway process or resource leak.

This is why we need a **multivariate approach** that learns the relationships between features.
Univariate thresholds cannot capture these cross-feature dependencies.

Reconstruction-based models like **TranAD** learn what "normal" combinations of feature values
look like. At inference time, the model reconstructs each timestep from context. When the
reconstruction error is high, it means the observed combination of values deviates from
learned normal patterns -- even if each individual feature value falls within its typical range.

## Summary

The SMD dataset provides rich multivariate structure with complex inter-feature dependencies.
Anomalies involve coordinated deviations across multiple dimensions -- exactly the scenario
where transformer-based reconstruction models like TranAD excel.